In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

## Template

This is the canonical PocketLM lesson shape: a **setup** cell, a **teach**
markdown cell, an **observe/TODO** code cell, and a final **assertion** cell so
the notebook fails loudly in CI if the engine breaks. Each u0 lesson copies this
structure.

In [ ]:
# Observe: the full PocketLM workflow at micro scale (load -> train -> generate).
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
sample = generate(model, tok, "The ", max_new_tokens=20, temperature=0.8, seed=0)
print(sample)

In [ ]:
# Assertion: CI catches breakage here.
assert len(sample) == len("The ") + 20
assert set(sample) <= set(tok.itos)